# 深度學習框架應用（大學部）｜主題 14：模型最佳化與壓縮

這本筆記本配合投影片《主題 14：模型最佳化與壓縮》。主角是 Fashion-MNIST（10 類衣服鞋子的 28×28 灰階照片）。我們先練一個 784 → 512 → 512 → 10 的 MLP 當基準，再用五題把它變小、變快，而且每一步都要量：換格式 → 動態量化 → 剪枝 → 知識蒸餾 → 效能基準。**練習一～五，每題 20%，合計 100%**。

| 練習 | 題目 | 投影片（練習頁） | 觀念在哪一頁 |
|---|---|---|---|
| 一 | 同一個模型，換格式有多大 | 第 110 頁 | 第 19～26 頁 |
| 二 | 動態量化前後比一比 | 第 111 頁 | 第 31～42 頁 |
| 三 | 全域剪枝，再微調救回來 | 第 112 頁 | 第 51～58 頁 |
| 四 | 知識蒸餾，比一比溫度 T | 第 113 頁 | 第 61～65 頁 |
| 五 | 自己寫效能基準，五個模型一起比 | 第 114 頁 | 第 14～17 頁 |

**怎麼用**

1. 用私人 Google 帳號登入（不使用學校帳號），按上方的「複製到雲端硬碟」（或「檔案 › 在雲端硬碟中儲存複本」）另存一份自己的複本，之後都改自己那份。**沒有另存就直接寫，分頁一關就全部不見。** 存好後先把檔名改成 `深度學習_W14_學號_姓名.ipynb`（左上角點檔名就能改）。
2. 由上往下跑：先跑「環境準備」「準備資料」「共用小工具」「準備基準模型」四格，之後每一題如果有「準備（直接執行）」那一格，也要先跑。每一題只有一格標 ✏️ 的程式格：把每個 `___` 換成答案再執行，提示就寫在同一行後面的註解。還沒把 `___` 換掉就執行，會出現紅字錯誤（例如 TypeError、AttributeError），錯誤訊息裡箭頭 `---->` 指的那一行就是還沒填的空格，填好再跑一次就好。
3. **這次不用切 GPU**：PyTorch 的動態量化只有 CPU 版，延遲也要在同一台 CPU 上比，所以全部在 CPU 上做。五題加起來 CPU 大約 3～5 分鐘。時間（毫秒、每秒幾張）每台機器都不一樣，**看倍數和趨勢就好**；準確率最後幾位也可能和投影片差一點點。改壞了就重新開原本的連結，再存一份複本。
4. **每題做完就截一張整個螢幕的圖（截圖 ①～⑤）**，要看得到身份指紋：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間，以及這一題的輸出。**看不到身份指紋，那一題不計分。**

**繳交**：做完照最後一格「繳交前檢查」，把**分享連結 1 個、.ipynb 檔 1 個、截圖 5 張（①～⑤）**交到 Flipclass 作業區。

## 環境準備

下面四格五題都會用到：載入套件、準備 Fashion-MNIST、幾個共用小工具（`MLP` 模型、`train` 訓練、`accuracy` 測試準確率、`size_mb` 量檔案大小），最後練一個基準模型 `teacher`。第一次跑會下載資料，大約 30 秒；基準模型在 CPU 上練 3 回合，大約 30 秒～1 分鐘。

In [ ]:
# ===== 環境準備（直接執行）=====
import copy, io, os, time, statistics, warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils.prune as prune
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from torchvision import datasets
warnings.filterwarnings('ignore')   # 量化工具會一直提醒「以後要換新套件」，先關掉
N_THREADS = torch.get_num_threads()   # 記下原本的執行緒數
print('PyTorch：', torch.__version__)
print('量化引擎：', torch.backends.quantized.engine, '｜這次全部在 CPU 上做')

In [ ]:
# ===== 準備資料：Fashion-MNIST（直接執行，不用改）=====
tr = datasets.FashionMNIST('data', train=True, download=True)
te = datasets.FashionMNIST('data', train=False, download=True)
norm = lambda d: ((d.data.float() / 255 - 0.2860) / 0.3530).unsqueeze(1)
Xtr, ytr = norm(tr), tr.targets      # 訓練 60,000 張
Xte, yte = norm(te), te.targets      # 測試 10,000 張
print('訓練：', tuple(Xtr.shape), '測試：', tuple(Xte.shape))

In [ ]:
# ===== 共用小工具（直接執行，不用改）=====
class MLP(nn.Module):                 # 和投影片「範例①」同一個模型
    def __init__(self, h=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(), nn.Linear(784, h), nn.ReLU(),
            nn.Linear(h, h), nn.ReLU(), nn.Linear(h, 10))
    def forward(self, x):
        return self.net(x)

def loader(n=60000, bs=128, seed=0):
    """取前 n 張訓練資料，每盒 bs 張；洗牌的順序固定"""
    ds = TensorDataset(Xtr[:n], ytr[:n])
    return DataLoader(ds, batch_size=bs, shuffle=True,
                      generator=torch.Generator().manual_seed(seed))

def train(model, epochs=3, n=60000, seed=0):
    """一般的訓練迴圈：Adam、學習率 1e-3"""
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for ep in range(epochs):
        model.train()
        for xb, yb in loader(n, seed=seed + ep):
            loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
    return model

@torch.no_grad()
def accuracy(model, dtype=torch.float32):
    """10,000 張測試照片的準確率；FP16 模型要把輸入也轉成 FP16"""
    model.eval()
    return (model(Xte.to(dtype)).argmax(1) == yte).float().mean().item()

def size_mb(model, path='tmp.pt'):
    """把 state_dict 存成檔案、量大小（MB，這裡 1 MB＝10⁶ bytes），量完就刪"""
    torch.save(model.state_dict(), path)
    mb = os.path.getsize(path) / 1e6
    os.remove(path)
    return mb

In [ ]:
# ===== 準備基準模型（直接執行，CPU 約 30 秒～1 分鐘）=====
torch.manual_seed(42)
teacher = train(MLP(512), epochs=3)   # FP32 基準，也是練習四的老師
print(f'基準模型（FP32，h＝512）測試準確率：{accuracy(teacher):.4f}')

## 練習一：同一個模型，換格式有多大 (20%)

> 題目在投影片**第 110 頁**｜觀念在第 19～26 頁

**情境**：小智練好了一個 784 → 512 → 512 → 10 的 MLP，想放進一台記憶體很小的機器。先別急著動模型，用算的就知道：每個權重改用比較少的位元存，檔案會剩多少？

**這題在練**：用「參數個數 × 每個佔幾個 byte」算出 FP32、FP16、INT8 的大小，再實測；順便看 FP16 和 BF16 存數字的差別

**步驟**

1. 用 `numel()` 數出模型一共有幾個參數（權重加偏差），再乘上每種格式每個數字佔幾個 byte
2. 用 `.half()` 把整個模型轉成 FP16，實測 FP32 和 FP16 兩個檔案各多大，也看 FP16 的準確率有沒有掉
3. 把 1/3 和 70000 分別存成 FP16、BF16，看看哪一個不準、哪一個直接爆掉

**要繳交**

- 截圖 ①：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ✏️ 練習一：同一個模型，換格式有多大
n = sum(p.___() for p in teacher.parameters())  # ✏️
print('參數量：', n)
B = {'FP32': 4, 'FP16': ___, 'INT8': 1}     # ✏️ 幾 byte
for fmt, b in B.items():
    print(f'{fmt} 算出來 {n * b / 1e6:.3f} MB')
half = copy.deepcopy(teacher).___()         # ✏️ 轉 FP16
print(f'實測 FP32 {size_mb(teacher):.3f} MB，'
      f'FP16 {size_mb(half):.3f} MB')
print('FP16 準確率：', accuracy(half, torch.float16))
x = torch.tensor([1 / 3, 70000.0])
for dt in [torch.float16, torch.___]:       # ✏️ BF16
    print(dt, x.to(dt).tolist())

#### 📸 截圖 ①

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：參數量、三行「算出來」的大小、實測的兩個大小、FP16 準確率，以及 FP16／BF16 那兩行

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 算出來的 FP32 是 2.679 MB，實測卻多一點點，多出來的是什麼？FP16 為什麼差不多剛好是一半？**

（在這裡寫）

**2. 70000 存成 FP16、BF16 各變成多少？為什麼？訓練大模型時為什麼常用 BF16？**

（在這裡寫）

## 練習二：動態量化前後比一比 (20%)

> 題目在投影片**第 111 頁**｜觀念在第 31～42 頁

**情境**：同一個基準模型，用一行 `quantize_dynamic` 把所有 Linear 層的權重換成 INT8，再量四個數字：檔案大小、準確率、單筆延遲、一次 64 筆的延遲。

**這題在練**：親手做一次訓練後量化（PTQ）裡最簡單的動態量化，看到 s、z 和整數權重，並量出「小了多少、掉了多少、快了多少」

**步驟**

1. 用 `torch.ao.quantization.quantize_dynamic` 把 `teacher` 的 `nn.Linear` 換成 INT8（原本的 teacher 不會被改到）
2. 印出第一層，看它變成什麼；再看這一層權重的 scale s、zero-point z 和前 5 個整數
3. 比較 FP32 和 INT8 的大小、準確率、單筆延遲、64 筆延遲

**要繳交**

- 截圖 ②：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習二：準備（直接執行，不用改）=====
torch.set_num_threads(1)    # 量時間時固定 1 個執行緒：比較公平，數字也比較穩
@torch.no_grad()
def latency_ms(m, bs=1, n=200):
    """一次推論 bs 張要幾毫秒：先暖機一次，再跑 n 次取平均"""
    x = torch.randn(bs, 1, 28, 28)
    m.eval(); m(x)                    # 暖機：第一次特別慢，不計時
    t0 = time.perf_counter()
    for _ in range(n):
        m(x)
    return (time.perf_counter() - t0) / n * 1000

In [ ]:
# ✏️ 練習二：動態量化，前後比一比
q_model = torch.ao.quantization.___(  # ✏️
    teacher, {nn.___}, dtype=torch.qint8)  # ✏️ 哪種層
print(q_model.net[1])        # 第一層變成什麼
w = q_model.net[1].weight()  # 量化後的權重
print('s =', w.q_scale(), '｜z =', w.q_zero_point())
print('前 5 個整數：', w.___()[0, :5].tolist())  # ✏️
for tag, m in [('FP32', teacher), ('INT8', q_model)]:
    print(f'{tag}  {size_mb(m):.3f} MB  '
          f'準確率 {accuracy(m):.4f}  '
          f'單筆 {latency_ms(m):.3f} ms  '
          f'64 筆 {latency_ms(m, bs=64):.3f} ms')

#### 📸 截圖 ②

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：第一層的名字、s 和 z 那一行、前 5 個整數，以及 FP32、INT8 兩行比較

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 第一層變成什麼？z 是多少、為什麼？量化後每個權重的誤差最多多大？**

（在這裡寫）

**2. 大小小了幾倍？準確率掉了多少？單筆和 64 筆各快了幾倍？為什麼檔案小了 4 倍，速度卻不一定快 4 倍？**

（在這裡寫）

## 練習三：全域剪枝，再微調救回來 (20%)

> 題目在投影片**第 112 頁**｜觀念在第 51～58 頁

**情境**：把三層 Linear 的 668,672 個權重放在一起比絕對值，剪掉最小的 50%、70%、90%，看準確率掉多少；再拿剪 90% 的那一個微調 1 回合，看救不救得回來。

**這題在練**：用 `torch.nn.utils.prune` 做全域非結構化剪枝，量稀疏度與準確率，並看到遮罩讓檔案變大、`prune.remove` 之後回到原大小

**步驟**

1. 先執行「練習三：準備」：`targets` 列出要剪的權重、`sparsity` 算稀疏度、`fine_tune` 微調
2. 在 ✏️ 格填好剪枝方法（看絕對值大小），剪 50%、70%、90% 各一次（每次都從原模型重新剪）
3. 剪 90% 那一個：看帶著遮罩的檔案大小 → 微調 1 回合 → `prune.remove` 定案，再量一次大小

> 微調要練 1 回合（60,000 張），CPU 上大約 10～30 秒。

**要繳交**

- 截圖 ③：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習三：準備（直接執行，不用改）=====
torch.set_num_threads(N_THREADS)    # 接下來要訓練，執行緒數還原
def targets(m):
    """剪枝名單：所有 Linear 層的 weight（偏差不剪）"""
    return [(l, 'weight') for l in m.modules() if isinstance(l, nn.Linear)]

def sparsity(m):
    """稀疏度：權重裡 0 佔多少比例"""
    z = sum((l.weight == 0).sum().item() for l, _ in targets(m))
    n = sum(l.weight.numel() for l, _ in targets(m))
    return z / n

def fine_tune(m, epochs=1):
    """剪完再練一下；遮罩會自動維持，剪掉的位置一直是 0"""
    return train(m, epochs=epochs, seed=10)

def report3(tag, m):
    print(f'{tag}：稀疏度 {sparsity(m):.1%}，準確率 {accuracy(m):.4f}')

In [ ]:
# ✏️ 練習三：全域剪枝，再微調救回來
for amt in [0.5, 0.7, 0.9]:
    m = copy.deepcopy(teacher)   # 每次都從原模型剪
    prune.global_unstructured(
        targets(m), amount=amt,
        pruning_method=prune.___)  # ✏️ 絕對值
    report3(f'剪 {amt:.0%}', m)  # 印稀疏度、準確率
print(f'帶著遮罩存檔：{size_mb(m):.2f} MB')  # 剪 90% 那個
fine_tune(m, epochs=___)         # ✏️ 微調 1 回合
report3('微調 1 回合後', m)
for mod, name in targets(m):
    prune.___(mod, name)         # ✏️ 定案
pruned = m                       # 練習五會用到
print(f'remove 之後：{size_mb(pruned):.2f} MB')

#### 📸 截圖 ③

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：三行「剪 50%／70%／90%」、帶著遮罩的大小、微調後那一行、remove 之後的大小

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 剪 50%、70%、90%（不微調）準確率各是多少？從哪裡開始掉很多？微調 1 回合後回到多少？**

（在這裡寫）

**2. 帶著遮罩的檔案為什麼是約 5.36 MB？remove 之後回到 2.68 MB：剪掉了 90%，檔案為什麼沒有變小？**

（在這裡寫）

## 練習四：知識蒸餾，比一比溫度 T (20%)

> 題目在投影片**第 113 頁**｜觀念在第 61～65 頁

**情境**：老師是剛剛練好的基準模型（h＝512），學生是 h＝64 的小 MLP，參數只有老師的十二分之一。學生只拿 6,000 張、練 3 回合，比較「只用硬標籤」和溫度 T＝1、2、4、8 的蒸餾。

**這題在練**：自己寫出蒸餾損失（軟目標乘 T² 加上硬目標），先用投影片的示範數字對答案，再看溫度怎麼影響學生

**步驟**

1. 在 ✏️ 格寫好 `kd_loss`：學生取 log 機率、老師取機率，算 KL 後乘上 T²，再和交叉熵照 alpha 加權
2. 用投影片那組示範 logits 算一次，應該得到 0.576 左右
3. 先執行「練習四：準備」再跑 ✏️ 格：只用硬標籤（alpha＝0）和 T＝1、2、4、8 各練一個學生，比較準確率

> 一共練 5 個學生模型，CPU 上大約 30 秒～1 分鐘。

**要繳交**

- 截圖 ④：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習四：準備（直接執行，不用改）=====
def distill(kd_loss, T, alpha=0.7, n=6000, epochs=3):
    """老師 teacher 教一個 h＝64 的學生：只用 6,000 張、練 3 回合"""
    torch.manual_seed(0)              # 每個學生都從同一組亂數開始
    student = MLP(64)
    opt = torch.optim.Adam(student.parameters(), lr=1e-3)
    teacher.eval()
    for ep in range(epochs):
        student.train()
        for xb, yb in loader(n, bs=64, seed=ep):
            with torch.no_grad():
                t_out = teacher(xb)   # 老師只出答案，不更新
            loss = kd_loss(student(xb), t_out, yb, T, alpha)
            opt.zero_grad(); loss.backward(); opt.step()
    return student, accuracy(student)

def report4(res):
    for k, (stu, acc) in res.items():
        print(f'{k:10} 學生準確率 {acc:.4f}')

zs = torch.tensor([[2.0, 0.5, 1.5]])  # 投影片的示範：學生的 logits
zt = torch.tensor([[4.0, 2.3, 1.0]])  # 老師的 logits（7、1、9 三類）
y0 = torch.tensor([0])                # 正確答案是第 0 類（7）
n_stu = sum(p.numel() for p in MLP(64).parameters())
print(f'學生 h＝64 的參數量：{n_stu:,}（老師的 1/{669706 / n_stu:.1f}）')

In [ ]:
# ✏️ 練習四：知識蒸餾，比一比溫度 T
def kd_loss(s, t, y, T, alpha=0.7):
    p_s = F.___(s / T, dim=-1)          # ✏️ 學生 log 機率
    p_t = F.softmax(t / T, dim=-1)      # 老師：機率
    kl = F.kl_div(p_s, p_t, reduction='batchmean')
    soft = kl * ___                     # ✏️ 補回 T²
    hard = F.cross_entropy(s, y)        # 硬目標
    return alpha * soft + (1 - alpha) * hard
print('示範：', round(kd_loss(zs, zt, y0, 4).item(), 3))
res = {'hard only': distill(kd_loss, T=1, alpha=0.0)}
for T in [1, 2, 4, 8]:
    res[f'T={T}'] = distill(kd_loss, T=___)  # ✏️
report4(res)                            # 印出準確率

#### 📸 截圖 ④

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：示範那一行（約 0.576），以及只用硬標籤和 T＝1、2、4、8 的準確率

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 示範那一行算出多少？拆成軟目標、硬目標兩項各是多少？如果把 `F.log_softmax` 寫成 `F.softmax` 會怎樣？**

（在這裡寫）

**2. 你的結果裡，哪一個 T 最好、哪一個最差？用「軟標籤變平」解釋 T＝8；蒸餾有沒有贏只用硬標籤？**

（在這裡寫）

## 練習五：自己寫效能基準，五個模型一起比 (20%)

> 題目在投影片**第 114 頁**｜觀念在第 14～17 頁

**情境**：前四題做出了五個模型：FP32 基準、INT8 動態量化、剪 90% 再微調、蒸餾學生、蒸餾學生再量化。最後用同一把尺、在同一台機器上一次比完。

**這題在練**：寫一個像樣的效能基準函式：先暖機、跑很多次取中位數，同時量延遲和吞吐量，再把大小、準確率、速度放進同一張表

**步驟**

1. 先執行「練習五：準備」：挑出蒸餾學生裡最好的那一個，再量化一份，五個模型放進 `models`
2. 在 ✏️ 格寫好 `benchmark`：暖機 10 次、量 100 次、取中位數，回傳（延遲毫秒, 每秒幾張）
3. 看表格最右邊三欄：每個模型比 FP32 小幾倍、單筆快幾倍、準確率掉幾個百分點

> 五個模型各量兩種 batch，CPU 上大約 10～30 秒。

**要繳交**

- 截圖 ⑤：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習五：準備（直接執行，不用改）=====
torch.set_num_threads(1)    # 又要量時間了：固定 1 個執行緒
best = max((k for k in res if k.startswith('T=')), key=lambda k: res[k][1])
student = res[best][0]                # 蒸餾學生裡準確率最高的那一個
s_int8 = torch.ao.quantization.quantize_dynamic(
    student, {nn.Linear}, dtype=torch.qint8)       # 蒸餾＋量化，兩招疊加
models = {'FP32': teacher, 'INT8': q_model, 'Pruned 90%': pruned,
          f'Student {best}': student, 'Student+INT8': s_int8}

ROWS5 = []

def report5(models, bench):
    """每個模型量大小、準確率、單筆延遲、64 筆吞吐量，印表（相對 FP32）再畫圖"""
    rows = [[k, size_mb(m), accuracy(m), *bench(m), bench(m, bs=64)[1]]
            for k, m in models.items()]
    f = rows[0]
    print(f"{'模型':14}{'大小MB':>8}{'準確率':>8}{'單筆ms':>8}{'64筆張/秒':>11}"
          f"{'小幾倍':>7}{'快幾倍':>7}{'掉幾個百分點':>8}")
    for r in rows:
        print(f'{r[0]:16}{r[1]:8.3f}{r[2]:9.4f}{r[3]:9.3f}{r[5]:12.0f}'
              f'{f[1] / r[1]:9.1f}×{f[3] / r[3]:8.2f}×{(f[2] - r[2]) * 100:10.2f}')
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    names = [r[0] for r in rows]
    ax[0].barh(names, [r[1] for r in rows]); ax[0].set_xlabel('size (MB)')
    ax[1].barh(names, [r[5] for r in rows]); ax[1].set_xlabel('throughput, batch 64 (images/s)')
    for a in ax: a.invert_yaxis(); a.grid(alpha=.3, axis='x')
    plt.tight_layout(); plt.show()
    ROWS5[:] = rows                   # 留一份，之後想再看可以印 ROWS5

In [ ]:
# ✏️ 練習五：自己寫效能基準函式，五個模型一起比
def benchmark(m, bs=1, n=100):
    x = torch.randn(bs, 1, 28, 28); m.eval()
    with torch.no_grad():
        for _ in range(___): m(x)      # ✏️ 暖機 10 次
        ts = []
        for _ in range(n):             # 一次一次計時
            t0 = time.perf_counter(); m(x)
            ts.append(time.perf_counter() - t0)
    ms = statistics.___(ts) * 1000     # ✏️ 中位數
    return ms, bs / ms * 1000          # 延遲、每秒幾張
print('FP32 單筆：', benchmark(teacher))
report5(models, benchmark)             # 五個模型一起量

#### 📸 截圖 ⑤

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：五行比較表（大小、準確率、單筆延遲、64 筆吞吐量、三個倍數）和兩張長條圖

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 為什麼要先暖機？為什麼取中位數，不取平均？**

（在這裡寫）

**2. 用你的表回答：「蒸餾＋INT8」的檔案比 FP32 小了幾倍？單筆延遲有沒有快那麼多倍？為什麼？如果要放進記憶體很小、又要即時反應的裝置，你選哪一個？**

（在這裡寫）

## AI 使用紀錄

有用 AI 就寫下來，沒用就寫「沒有使用」。

| 用了哪個 AI | 問了什麼（大概就好） | 它哪裡幫到你 | 它哪裡說錯、你怎麼發現的 |
|---|---|---|---|
| （例：ChatGPT） | （例：kl_div 的兩個參數順序） |  |  |

## 繳交前檢查

1. **重跑一次**：「執行階段 › 重新啟動工作階段並執行所有儲存格」，每一格都有輸出、沒有紅字。
2. **檔名**：`深度學習_W14_學號_姓名.ipynb`（左上角點檔名就能改）。
3. **截圖**：5 張（①～⑤），每張都是整個螢幕，看得到身份指紋（左上的檔名、右上的 Google 頭貼、右下工作列的日期時間）和那一題的輸出。
4. **開放分享**：右上角「分享」→「一般存取權」改成「知道連結的任何人」→ 角色選「檢視者」→「複製連結」。
5. **自己檢查**：開一個無痕視窗貼上連結，不用登入就打得開才算成功。
6. **下載**：「檔案 › 下載 › 下載 .ipynb」。
7. **交到 Flipclass**：分享連結 1 個、.ipynb 檔 1 個、截圖 5 張（①～⑤），交到 Flipclass 作業區，期限照 Flipclass 規定。